In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("student-por.csv", sep=";")

In [3]:
df.shape

(649, 33)

In [4]:
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      649 non-null    object
 1   sex         649 non-null    object
 2   age         649 non-null    int64 
 3   address     649 non-null    object
 4   famsize     649 non-null    object
 5   Pstatus     649 non-null    object
 6   Medu        649 non-null    int64 
 7   Fedu        649 non-null    int64 
 8   Mjob        649 non-null    object
 9   Fjob        649 non-null    object
 10  reason      649 non-null    object
 11  guardian    649 non-null    object
 12  traveltime  649 non-null    int64 
 13  studytime   649 non-null    int64 
 14  failures    649 non-null    int64 
 15  schoolsup   649 non-null    object
 16  famsup      649 non-null    object
 17  paid        649 non-null    object
 18  activities  649 non-null    object
 19  nursery     649 non-null    object
 20  higher    

In [6]:
df.duplicated().sum()

0

In [7]:
df["G3"].describe()

count    649.000000
mean      11.906009
std        3.230656
min        0.000000
25%       10.000000
50%       12.000000
75%       14.000000
max       19.000000
Name: G3, dtype: float64

In [8]:
df["Pass_Fail"] = (df["G3"] >= 10).map({True: "Pass", False: "Fail"})
df["Pass_Fail"].value_counts()

Pass_Fail
Pass    549
Fail    100
Name: count, dtype: int64

Define Features and Target

In [9]:
y = df["Pass_Fail"]
X = df.drop(columns=["Pass_Fail", "G2", "G3"])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (649, 31)
y shape: (649,)


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print(y_test.value_counts())

Training set: (519, 31)
Test set: (130, 31)
Pass_Fail
Pass    110
Fail     20
Name: count, dtype: int64


Train a Logistic Regression Mode

In [11]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), make_column_selector(dtype_include="number")),
    ("categorical", OneHotEncoder(handle_unknown="ignore"),
     make_column_selector(dtype_exclude="number")),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)
print("Model trained successfully.")

Model trained successfully.


Evaluate the Logistic Regression Model

In [12]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)
labels = ["Pass", "Fail"]

print(classification_report(y_test, y_pred, labels=labels, zero_division=0))
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, y_pred, labels=labels))

              precision    recall  f1-score   support

        Pass       0.89      0.92      0.91       110
        Fail       0.47      0.40      0.43        20

    accuracy                           0.84       130
   macro avg       0.68      0.66      0.67       130
weighted avg       0.83      0.84      0.83       130

Confusion matrix (rows = actual, columns = predicted):
[[101   9]
 [ 12   8]]
